# 02. ETL: Поведенческие агрегаты

> **Краткое резюме**: Вычисляем поведенческие признаки для каждого клиента: обороты, количество транзакций, направленность потоков, динамика контрагентов, остатки, продукты. Результат — `data/behavioral_features.parquet`.

**Источники**:
- `paymentcounteragent_stran` — транзакции (основной)
- `clientbalance_sagg` — остатки на счетах
- `bmbpenetrproduct_stat` — продуктовое проникновение

**Период**: Полный год (Jan–Dec 2025). Первая половина: Jan–Jun, вторая: Jul–Dec.

**Предпосылки**: Запустите `01_etl_profile.ipynb` (создаёт вселенную клиентов).

---

In [ ]:
import logging
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
log = logging.getLogger(__name__)

In [ ]:
# =====================================================
# ПАРАМЕТРЫ (должны совпадать с 01_etl_profile)
# =====================================================
HIVE_DATABASE = "s_dmrb"

# Полный год: Jan–Dec 2025
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
START_INT = int(START_DATE.replace("-", ""))
END_INT = int(END_DATE.replace("-", ""))

MID_DATE = "2025-07-01"
MID_INT = int(MID_DATE.replace("-", ""))

# Pilot: 1% — тот же hash что в 01, ~144К клиентов.
# Для прода: SAMPLE_PCT = 100
SAMPLE_PCT = 1

OUTPUT_DIR = os.path.abspath("./data")
TMP_DIR = os.path.expanduser("~/tmp")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
SPARK_OUTPUT_DIR = f"file://{OUTPUT_DIR}"

print(f"Period : {START_DATE} — {END_DATE}, mid: {MID_DATE}")
print(f"Sample : {SAMPLE_PCT}%  (~{14_400_000 * SAMPLE_PCT // 100:,} clients est.)")

In [ ]:
try:
    _ = spark.sparkContext.appName  # noqa: F821
    print(f"SparkSession available: {spark.sparkContext.appName}")  # noqa: F821
except Exception:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("LookAlike_ETL_Behavior").enableHiveSupport().getOrCreate()
    print("SparkSession created")

# Spill в ~/tmp — /mnt/datastore на 100% занят
spark.conf.set("spark.local.dir", TMP_DIR)  # noqa: F821
# Мелкие блоки чтения снижают heap-давление на executor при GROUP BY
spark.conf.set("spark.sql.files.maxPartitionBytes", "32m")
# 200 shuffle-партиций — дефолт, явно фиксируем
spark.conf.set("spark.sql.shuffle.partitions", "200")

spark.sql(f"USE {HIVE_DATABASE}")  # noqa: F821
print(f"Spark {spark.version}, database: {HIVE_DATABASE}")  # noqa: F821

---
## 1. Загрузка вселенной из 01_etl_profile

In [ ]:
profiles_path = f"{SPARK_OUTPUT_DIR}/client_profiles.parquet"
profiles_df = spark.read.parquet(profiles_path)
profiles_df.select("client_uk").createOrReplaceTempView("universe")
print(f"Universe loaded: {profiles_df.count():,} clients")

---
## 2. Транзакционные агрегаты (`paymentcounteragent_stran`)

Основной источник поведенческих фичей. Каждая транзакция записана с точки зрения `client_uk`:
- `income_flag = 'N'` → клиент платит (outflow)
- `income_flag = 'Y'` → клиент получает (inflow)

In [ ]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "32m")
spark.conf.set("spark.sql.shuffle.partitions", "500")

sample_filter = f"AND ABS(HASH(p.client_uk)) % 100 < {SAMPLE_PCT}" if SAMPLE_PCT < 100 else ""

tx_agg_query = spark.sql(f"""
    SELECT
        p.client_uk,
        COUNT(*) AS tx_count,
        SUM(p.rur_amt) AS total_amount,
        SUM(CASE WHEN p.income_flag = 'N' THEN p.rur_amt ELSE 0 END) AS total_outflow,
        SUM(CASE WHEN p.income_flag = 'Y' THEN p.rur_amt ELSE 0 END) AS total_inflow,
        AVG(p.rur_amt) AS avg_tx_amount,
        STDDEV(p.rur_amt) AS std_tx_amount,
        approx_count_distinct(p.client_contr_uk) AS unique_counterparties,
        approx_count_distinct(
            CASE WHEN p.income_flag = 'N' THEN p.client_contr_uk END
        ) AS unique_payees,
        approx_count_distinct(
            CASE WHEN p.income_flag = 'Y' THEN p.client_contr_uk END
        ) AS unique_payers,
        COUNT(DISTINCT CAST(FLOOR(p.date_part / 100) AS INT)) AS active_months,
        SUM(CASE WHEN p.date_part < {MID_INT} THEN p.rur_amt ELSE 0 END) AS amount_first_half,
        SUM(CASE WHEN p.date_part >= {MID_INT} THEN p.rur_amt ELSE 0 END) AS amount_second_half,
        approx_count_distinct(
            CASE WHEN p.date_part < {MID_INT} THEN p.client_contr_uk END
        ) AS cp_first_half,
        approx_count_distinct(
            CASE WHEN p.date_part >= {MID_INT} THEN p.client_contr_uk END
        ) AS cp_second_half
    FROM {HIVE_DATABASE}.paymentcounteragent_stran p
    WHERE p.date_part >= {START_INT} AND p.date_part <= {END_INT}
      AND (p.deleted_flag IS NULL OR p.deleted_flag != 'Y')
      AND p.rur_amt IS NOT NULL AND p.rur_amt > 0
      AND p.client_uk IS NOT NULL
      {sample_filter}
    GROUP BY p.client_uk
""")

tx_agg_path = f"{SPARK_OUTPUT_DIR}/tx_agg.parquet"
tx_agg_query.write.mode("overwrite").parquet(tx_agg_path)
print("Transaction aggregates written to disk")

tx_agg_df = spark.read.parquet(tx_agg_path)
tx_agg_df.createOrReplaceTempView("tx_universe")
print(f"Transaction aggregates: {tx_agg_df.count():,} clients")
tx_agg_df.describe("total_amount", "tx_count", "unique_counterparties", "active_months").show()

---
## 3. Остатки на счетах (`clientbalance_sagg`)

Средний / макс / мин остаток за период. Даёт представление о «размере» клиента.

In [ ]:
# Верификация колонок (раскомментируйте при первом запуске)
# spark.sql(f'DESCRIBE {HIVE_DATABASE}.clientbalance_sagg').show(60, truncate=False)

In [ ]:
if not spark.catalog.tableExists("tx_universe"):
    _tx_path = f"{SPARK_OUTPUT_DIR}/tx_agg.parquet"
    spark.read.parquet(_tx_path).createOrReplaceTempView("tx_universe")
    print("tx_universe view restored from parquet")

try:
    _parts = spark.sql(f"SHOW PARTITIONS {HIVE_DATABASE}.clientbalance_sagg").collect()
    _last_part = max(int(r[0].split("=")[1]) for r in _parts if int(r[0].split("=")[1]) <= END_INT)
    print(f"Using balance snapshot: date_part={_last_part}")

    sample_filter_b = (
        f"AND ABS(HASH(CAST(b.client_uk AS BIGINT))) % 100 < {SAMPLE_PCT}"
        if SAMPLE_PCT < 100
        else ""
    )
    balance_path = f"{SPARK_OUTPUT_DIR}/balance_agg.parquet"
    spark.sql(f"""
        SELECT
            CAST(b.client_uk AS BIGINT)    AS client_uk,
            AVG(b.balance_rur_amt)         AS last_balance,
            AVG(b.balance_rur_amt_30_avg)  AS avg_balance_30d,
            MAX(b.balance_rur_amt_30_max)  AS max_balance_30d,
            MIN(b.balance_rur_amt_30_min)  AS min_balance_30d
        FROM {HIVE_DATABASE}.clientbalance_sagg b
        INNER JOIN tx_universe u ON CAST(b.client_uk AS BIGINT) = u.client_uk
        WHERE b.date_part = {_last_part}
          {sample_filter_b}
        GROUP BY b.client_uk
    """).write.mode("overwrite").parquet(balance_path)
    balance_df = spark.read.parquet(balance_path)
    print(f"Balance aggregates: {balance_df.count():,} clients")
except Exception as e:
    print(f"Balance extraction failed: {e}")
    print("Skipping balance features.")
    balance_df = None

---
## 4. Продуктовое проникновение (`bmbpenetrproduct_stat`)

In [ ]:
if not spark.catalog.tableExists("tx_universe"):
    _tx_path = f"{SPARK_OUTPUT_DIR}/tx_agg.parquet"
    spark.read.parquet(_tx_path).createOrReplaceTempView("tx_universe")
    print("tx_universe view restored from parquet")

try:
    sample_filter_p = (
        f"AND ABS(HASH(CAST(pp.client_uk AS BIGINT))) % 100 < {SAMPLE_PCT}"
        if SAMPLE_PCT < 100
        else ""
    )
    product_path = f"{SPARK_OUTPUT_DIR}/product_agg.parquet"
    spark.sql(f"""
        SELECT
            CAST(pp.client_uk AS BIGINT)             AS client_uk,
            COUNT(DISTINCT pp.product_uk)            AS product_count,
            COUNT(DISTINCT pp.producttypemass_name)  AS product_type_count,
            SUM(pp.rur_amt)                          AS product_total_amt
        FROM {HIVE_DATABASE}.bmbpenetrproduct_stat pp
        INNER JOIN tx_universe u ON CAST(pp.client_uk AS BIGINT) = u.client_uk
        WHERE pp.date_part >= {START_INT} AND pp.date_part <= {END_INT}
          {sample_filter_p}
        GROUP BY pp.client_uk
    """).write.mode("overwrite").parquet(product_path)
    product_df = spark.read.parquet(product_path)
    print(f"Product aggregates: {product_df.count():,} clients")
except Exception as e:
    print(f"Product extraction failed: {e}")
    print("Skipping product features.")
    product_df = None

---
## 5. Сборка поведенческих фичей

In [ ]:
behavior_df = tx_agg_df

if balance_df is not None:
    behavior_df = behavior_df.join(balance_df, on="client_uk", how="left")

if product_df is not None:
    behavior_df = behavior_df.join(product_df, on="client_uk", how="left")

print(f"Columns: {behavior_df.columns}")

In [ ]:
# Сохранение — всё материализуется здесь одним проходом
out_path = f"{SPARK_OUTPUT_DIR}/behavioral_features.parquet"
behavior_df.write.mode("overwrite").parquet(out_path)

# Перечитываем с диска для статистики
behavior_df = spark.read.parquet(out_path)
print(f"Saved: {out_path}")
print(f"Behavioral features: {behavior_df.count():,} rows, {len(behavior_df.columns)} columns")

---
## 6. Статистика

In [ ]:
behavior_df.describe(
    "total_amount",
    "total_outflow",
    "total_inflow",
    "tx_count",
    "unique_counterparties",
    "active_months",
    "avg_tx_amount",
).show(truncate=False)

behavior_df.show(5, truncate=20, vertical=True)

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **total_outflow / total_inflow** | Сумма исходящих / входящих платежей за период |
| **unique_counterparties** | Количество уникальных контрагентов |
| **unique_payees / payers** | Уникальные получатели / плательщики |
| **active_months** | Количество месяцев с хотя бы 1 транзакцией |
| **amount_first_half / second_half** | Обороты в первой / второй половине периода (для тренда) |
| **cp_first_half / second_half** | Количество контрагентов в первой / второй половине |
| **avg_balance** | Средний остаток на счетах за период (RUR) |
| **product_count** | Количество уникальных банковских продуктов |

---

**Следующий шаг**: `03_feature_engineering.ipynb`.